In [ ]:
from pathlib import Path
import hashlib
import json
import re

import numpy as np
import pandas as pd
from IPython.display import display

from shared.common import load_config

In [ ]:
raw = Path('../input/prairie_grass')
out = Path('../output')
config = load_config()
raw_names = ['PGARCS.txt', 'PGVertConc.txt', 'PGrassTABLE_5_4.TXT',
             'PGrassTTUU.txt', 'PG.DAT', 'pgrprof.222', 'wind_10min_table61.csv']
radii = [50, 100, 200, 400, 800]

In [ ]:
# table 5.2 via Irwin/D6589 polar blocks
lines = iter((raw / 'PGARCS.txt').read_text().splitlines()[1:])
release_rows, arc_rows = [], []
for line in lines:
    if not line.strip():
        continue
    run, arc, date, window, radius = [item.strip(" '") for item in line.split(',')]
    run, arc, radius = int(run), int(arc), float(radius)
    n, first, last, polar, distance_factor, sx, sy, q, h, z, factor = map(float, next(lines).split(','))
    assert (polar, distance_factor, sx, sy, factor) == (1, 1, 0, 0, 1)
    release_rows.append((run, date, window, q, h))
    for index in range(1, int(n) + 1):
        bearing, post, distance, raw_c = map(float, next(lines).split())
        assert distance == radius == radii[arc - 1]
        if first <= index <= last:
            arc_rows.append((run, radius, bearing % 360, z, raw_c, 'arc'))

releases = pd.DataFrame(release_rows, columns=['run', 'date', 'window_local', 'q_g_s', 'h_m']).drop_duplicates()
releases['date'] = pd.to_datetime(releases['date'], format='%m/%d/%y').dt.date
assert releases['run'].is_unique
releases = releases.sort_values('run')

In [ ]:
# table 5.3 tower blocks
tower_rows = []
for line in (raw / 'PGVertConc.txt').read_text().splitlines():
    fields = line.split()
    if fields[:2] == ['Run', 'No.']:
        run = int(fields[2])
    elif fields and fields[0] == 'Hgt(m)':
        bearings = [int(value) for value in fields[1:]]
    elif len(fields) == 7 and re.fullmatch(r'\d+\.\d+', fields[0]):
        z, *values = map(float, fields)
        tower_rows.extend((run, 100, bearing, z, c, 'tower') for bearing, c in zip(bearings, values))
assert len(tower_rows) % 54 == 0

In [ ]:
# table 5.4 correction factors belong to horizontal arc aggregates
corrections = []
for line in (raw / 'PGrassTABLE_5_4.TXT').read_text().splitlines():
    if line[:8].strip().isdigit():
        for j, radius in enumerate(radii):
            value = line[8 + 9*j:17 + 9*j].strip()
            if value:
                corrections.append((int(line[:8]), radius, float(value)))
corr = pd.DataFrame(corrections, columns=['run', 'radius_m', 'arc_factor'])
assert corr.arc_factor.between(0.0, 1.01, inclusive="right").all()

obs = pd.DataFrame(arc_rows + tower_rows, columns=['run', 'radius_m', 'bearing_deg', 'z_m', 'raw_mg_m3', 'array'])
obs = obs.merge(corr, on=['run', 'radius_m'], how='left', validate='many_to_one')
obs.loc[obs['array'].eq('tower'), 'arc_factor'] = np.nan
obs['observed_mg_m3'] = obs['raw_mg_m3'].where(obs['raw_mg_m3'] >= 0) * obs['arc_factor'].fillna(1)
obs['obs_id'] = np.arange(len(obs))
assert set(obs.loc[obs['raw_mg_m3'] < 0, 'raw_mg_m3']) == {-9.0}
check = obs.query("run == 7 and radius_m == 50 and array == 'arc'")['observed_mg_m3'].max()
assert np.isclose(check, 92.91)

In [ ]:
# EPA evaluated releases + surface direction from matched profile key
met_rows = []
for line in (raw / 'PG.DAT').read_text().splitlines():
    if line[:5].strip().isdigit():
        run, date, hour, u2, sigtheta = line[:32].split()
        met_rows.append((int(run), date, int(hour) // 100, float(u2), float(sigtheta),
                         int(line[96:99]), float(line[45:52])))
met = pd.DataFrame(met_rows, columns=['run', 'epa_date', 'epa_hour', 'u2_m_s', 'sigma_theta_deg',
                                      'stability', 'epa_q_g_s'])
met['epa_date'] = pd.to_datetime(met['epa_date'], format='%m-%d-%Y').dt.date
met['met_day'] = met['epa_date'].where(~met['run'].isin([61, 62]), pd.Timestamp('1956-08-27').date())

epa_profile = pd.read_csv(raw / 'pgrprof.222', sep=r'\s+', header=None,
                          usecols=[0, 1, 2, 3, 4, 6], names=['yy', 'month', 'day', 'epa_hour', 'z_m', 'wind_from_deg'])
surface = epa_profile.loc[epa_profile['wind_from_deg'].between(0, 360)].copy()
surface['met_day'] = pd.to_datetime(surface[['month', 'day']].assign(year=surface['yy'] + 1900)).dt.date
met = met.merge(surface[['met_day', 'epa_hour', 'wind_from_deg']],
                on=['met_day', 'epa_hour'], how='left', validate='one_to_one')
assert met['wind_from_deg'].notna().all() and len(met) == 44
releases = releases.merge(met, on='run', how='left', validate='one_to_one')
# run 43: the original Table 5.1 agrees with the polar header (98.9)
q_gap = (releases['q_g_s'] - releases['epa_q_g_s']).abs().dropna()
assert np.allclose(q_gap[releases.loc[q_gap.index, 'run'].ne(43)], 0)
assert np.isclose(q_gap[releases.loc[q_gap.index, 'run'].eq(43)].iat[0], 0.3)

In [ ]:
# tower wind magnitude for shear; source U2 retained as model anchor
profile_rows = []
for line in (raw / 'PGrassTTUU.txt').read_text().splitlines():
    fields = line.split()
    if len(fields) == 2 and fields[0] == 'Exp':
        run = int(fields[1])
    elif len(fields) == 3 and re.fullmatch(r'\d+\.\d+', fields[0]):
        height, temp, speed = map(float, fields)
        profile_rows.append((run, height, temp if temp > -999 else np.nan, speed if speed > 0 else np.nan))
profiles = pd.DataFrame(profile_rows, columns=['run', 'height_m', 'temp_c', 'speed_m_s'])
assert profiles.groupby('run').size().eq(7).all()

In [ ]:
# table 6.1, handwritten archive values + documented downwind vane correction
wind = pd.read_csv(raw / 'wind_10min_table61.csv')
assert wind['run'].is_unique and set(wind['run']) == set(met['run'])
assert wind[['source_speed_m_s', 'source_from_deg', 'source_sd_deg']].notna().all().all()
assert wind['source_speed_m_s'].gt(0).all()
wind['downwind_corrected_from_deg'] = (wind['downwind_from_deg'] - config['alignment']['downwind_vane_bias_deg']) % 360
wind.to_csv(out / 'surface_winds.csv', index=False)

In [ ]:
releases.to_csv(out / 'releases.csv', index=False)
obs.to_csv(out / 'observations.csv', index=False)
profiles.to_csv(out / 'wind_profiles.csv', index=False)
(out / 'model_config.json').write_text(json.dumps(config, indent=2) + '\n')
manifest = pd.DataFrame([{'file': name, 'bytes': (raw / name).stat().st_size,
                          'sha256': hashlib.sha256((raw / name).read_bytes()).hexdigest()} for name in raw_names])
manifest.to_csv(out / 'source_manifest.csv', index=False)
display(manifest)
display(obs.groupby('array', as_index=False).agg(
    release_count=('run', 'nunique'), source_rows=('obs_id', 'size'),
    usable_rows=('observed_mg_m3', 'count'), arc_adjusted=('arc_factor', 'count')))